# Factura → tabla recortada → OCR

```
imagen original
   ↓  (1) LLM de visión  →  bounding box de la tabla de items
imagen recortada + zoom
   ↓  (2) PaddleOCR      →  texto + coordenadas
   ↓  (3) agrupado por y →  filas / DataFrame
```

**Este notebook no tiene lógica.** Todo vive en `pipeline.py` del repo.
Para cambiar el comportamiento se edita ese archivo, se pushea, y acá alcanza
con volver a correr la celda de *setup*.

> Colab: **Entorno de ejecución → Cambiar tipo → GPU T4** (opcional, más rápido).


## 0. Setup — traer el repo e instalar

Corré esta celda cada vez que haya cambios en `pipeline.py`. El `git pull` y el
`reload` hacen que los cambios entren sin reiniciar el entorno.


In [ ]:
REPO_URL = "https://github.com/FedericoRojo/imageProccesing.git"
CARPETA = "tesis-facturas"

import subprocess, sys
from pathlib import Path

# Repo privado -> cargá un token de GitHub en los secrets de Colab como
# GITHUB_TOKEN (fine-grained, permiso Contents: read).
# Si hacés el repo público, no hace falta ningún token.
tok = None
try:
    from google.colab import userdata
    tok = userdata.get("GITHUB_TOKEN")
except Exception:
    pass

url = REPO_URL.replace("https://", f"https://{tok}@") if tok else REPO_URL


def _correr(cmd):
    r = subprocess.run(cmd, capture_output=True, text=True)
    salida = (r.stdout + r.stderr)
    if tok:
        salida = salida.replace(tok, "***")   # que el token no quede en el output
    if r.returncode:
        raise RuntimeError(salida)
    return salida


if Path(CARPETA).exists():
    print(_correr(["git", "-C", CARPETA, "pull", "--ff-only"]))
else:
    _correr(["git", "clone", "--depth", "1", url, CARPETA])
    print("clonado")

sys.path.insert(0, str(Path(CARPETA).resolve()))
print("repo listo")

In [ ]:
# Instalación (sólo la primera vez del entorno; después es instantánea)
!pip install -q -r {CARPETA}/requirements.txt
print("dependencias listas")

In [ ]:
# Importar / recargar el pipeline. Correr después de cada git pull,
# y también si Colab reinició el entorno (se pierde el sys.path del setup).
import sys
from pathlib import Path

CARPETA = "tesis-facturas"
_ruta = str(Path(CARPETA).resolve())
if not Path(_ruta).exists():
    raise RuntimeError(f"No encuentro {CARPETA}/. Corré primero la celda de setup.")
if _ruta not in sys.path:
    sys.path.insert(0, _ruta)

import importlib
import pipeline
importlib.reload(pipeline)

print("pipeline listo:", [n for n in dir(pipeline) if not n.startswith("_")])

## 1. Entrada — subir la factura

In [ ]:
from pathlib import Path
from google.colab import files

uploaded = files.upload()              # .png / .jpg / .jpeg
IMG = Path(list(uploaded.keys())[0])
print("Archivo:", IMG)

# Si el archivo ya está en /content:
# IMG = Path("/content/factura2.jpeg")

## 2. Imagen → LLM → tabla recortada

Necesita `NVIDIA_API_KEY` en los secrets de Colab (ícono de la llave, panel izquierdo).


In [ ]:
from PIL import Image
from IPython.display import display

IMG_TABLA = pipeline.preparar_tabla(IMG)      # detecta el bbox y recorta
print("Tabla recortada:", IMG_TABLA)

display(Image.open(IMG_TABLA))                # revisá el recorte antes de seguir

Si el recorte quedó mal:

- corta filas → `pipeline.preparar_tabla(IMG, margen=0.05)`
- texto chico → `pipeline.preparar_tabla(IMG, zoom=3)`
- el LLM no encuentra nada → devuelve la imagen original y el pipeline sigue igual


## 3. Tabla recortada → OCR

In [ ]:
res = pipeline.ocr_tabla(IMG_TABLA)
print(len(res["rec_texts"]), "cajas de texto")

# Lo que peor leyó, primero — útil para detectar dónde falla
pipeline.texto_con_score(res).head(15)

In [ ]:
# Imagen anotada con las cajas detectadas
import glob
from IPython.display import Image as IPyImage, display

for p in glob.glob("output/*.jpg") + glob.glob("output/*.png"):
    display(IPyImage(p))

## 4. OCR → filas

`agrupar_filas` calcula la tolerancia vertical a partir del alto mediano de las
cajas, así que no hay que reajustarla cuando cambia el `zoom` del recorte.


In [ ]:
filas = pipeline.agrupar_filas(res)
pipeline.imprimir_filas(filas)

In [ ]:
df = pipeline.filas_a_dataframe(filas)
df

*(siguiente: mapear cada fila a ARTICULO / CANT. / DESCRIPCIÓN / PRECIO / IMPORTE
usando los cortes de columna, y validar `cant × precio ≈ importe`.)*


---

## Antes de commitear este notebook

`Editar → Borrar todos los resultados`. Con los outputs del OCR embebidos el
archivo pesa 20x más y los diffs se vuelven ilegibles.
